# Lesson 5.3 — Case Study: Reading Telemetry

Read a healthy run, a disturbed run, and a near-singular run side by side. Different signals move in each — effort/error for a disturbance, manipulability for a near-singular path.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, plan_reference, execute_reference, track,
    system_monitor, fk_xy, P2, T2)
def pick_layer(seed_world=1, seed_perc=7):
    w = GreenhouseWorld.demo_row(n=6, seed=seed_world)
    dets = model_perception(w, rng=np.random.default_rng(seed_perc))
    _, tgt = understand(dets, w)
    layer = plan_reference(w.q.copy(), to_configuration(tgt), rng=np.random.default_rng(0))
    return w, tgt, layer
checks = []
def read(rec, tgt):
    v = track(rec, target=tgt); m = system_monitor(rec)
    return v, m
w, tgt, layer = pick_layer()


### Run 1 (healthy) and Run 2 (external disturbance)

In [2]:
v1, m1 = read(execute_reference(layer, telemetry=True), tgt)
def big(t,j): return 60.0 if (j==0 and t>0.3) else 0.0
v2, m2 = read(execute_reference(layer, disturbance=big, telemetry=True), tgt)
print('healthy   : %s  peak_err=%.4f peak_effort=%.2f minW=%.4f' % (v1['reason'], m1['peak_error'], m1['peak_effort'], m1['min_manipulability']))
print('disturbed : %s  peak_err=%.3f peak_effort=%.2f minW=%.4f' % (v2['reason'], m2['peak_error'], m2['peak_effort'], m2['min_manipulability']))
# reading: disturbance -> error & effort spike, manipulability ~unchanged (external strain)
checks.append(v1['success'] and not v2['success'])
checks.append(m2['peak_effort'] > 3*m1['peak_effort'])
checks.append(abs(m2['min_manipulability'] - m1['min_manipulability']) < 0.02)  # geometry unchanged


healthy   : ok  peak_err=0.0001 peak_effort=4.61 minW=0.0861
disturbed : final_error  peak_err=2.203 peak_effort=70.02 minW=0.0861


### Run 3 (near-singular path): manipulability is the signal that moves

In [3]:
ns_tgt = {'xy': np.array([0.698, 0.0]), 'ripe': True, 'reachable': True}
ns_layer = plan_reference(np.array([0.3,0.9]), to_configuration(ns_tgt), rng=np.random.default_rng(0))
v3, m3 = read(execute_reference(ns_layer, telemetry=True), ns_tgt)
print('near-sing : %s  peak_err=%.4f peak_effort=%.2f minW=%.4f' % (v3['reason'], m3['peak_error'], m3['peak_effort'], m3['min_manipulability']))
# reading: manipulability drops sharply vs healthy -> geometry/planning condition, not a disturbance
checks.append(m3['min_manipulability'] < 0.5*m1['min_manipulability'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


near-sing : ok  peak_err=0.0001 peak_effort=3.59 minW=0.0183
4/4 checks passed.
All checks passed.
